# Day 5 — Pipeline Automation & Data Quality

This notebook demonstrates the end-to-end Heat-Pulse pipeline in:
1. **Full mode** — re-ingest everything, clean, engineer features.
2. **Incremental mode** — detect that data is already up-to-date and skip gracefully.
3. **Architecture diagram** — a text-based flowchart of the pipeline.


## 0  Imports & path setup

In [2]:
import sys, os
from pathlib import Path

# Make sure the src/ folder is importable
PROJECT_ROOT = Path.cwd().parent   # adjust if you run from a different folder
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Paths used throughout
DB_PATH   = PROJECT_ROOT / 'data' / 'weather.duckdb'
DATA_DIR  = PROJECT_ROOT / 'data'
LOG_DIR   = PROJECT_ROOT / 'logs'

print('Project root:', PROJECT_ROOT)
print('DB path     :', DB_PATH)


Project root: c:\Users\Vito\Desktop\Heat-Pulse  clone
DB path     : c:\Users\Vito\Desktop\Heat-Pulse  clone\data\weather.duckdb


## 1  Full pipeline run

Equivalent to:
```bash
python src/pipeline.py --mode full
```


In [3]:
from pipeline import run_pipeline

summary_full = run_pipeline(
    mode       = 'full',
    db_path    = DB_PATH,
    data_dir   = DATA_DIR,
    log_dir    = LOG_DIR,
    start_date = '2020-01-01',
)

print('\n── Full run summary ──────────────────────')
for k, v in summary_full.items():
    print(f'  {k:<20}: {v}')


CSV not found — using fallback city list.
[Nakhchivan] Attempt 1 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=39.209&longitude=45.4122&daily=temperature_2m_max%2Ctemperature_2m_min%2Ctemperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Cwind_speed_10m_max%2Cwind_gusts_10m_max%2Cpressure_msl_mean%2Cshortwave_radiation_sum%2Capparent_temperature_max%2Cweather_code&timezone=UTC&start_date=2020-01-01&end_date=2026-04-30
[Nakhchivan] Attempt 2 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=39.209&longitude=45.4122&daily=temperature_2m_max%2Ctemperature_2m_min%2Ctemperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Cwind_speed_10m_max%2Cwind_gusts_10m_max%2Cpressure_msl_mean%2Cshortwave_radiation_sum%2Capparent_temperature_max%2Cweather_code&timezone=UTC&start_date=2020-01-01&end_date=2026-04-30
[Nakhchivan] Attempt 3 failed: 429 Client Error:

[clean_raw_to_staging] data_dir = c:\Users\Vito\Desktop\Heat-Pulse  clone\data

  HISTORICAL  →  staging_historical
  Found parquet: c:\Users\Vito\Desktop\Heat-Pulse  clone\data\raw.parquet
  Rows: 9,248  |  Columns: ['time', 'city', 'latitude', 'longitude', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'pressure_msl_mean', 'shortwave_radiation_sum', 'apparent_temperature_max', 'weather_code']
  After cleaning: 9,248 rows
[flag_outliers] temperature_2m_max: 0 outliers flagged (0.00%)
[flag_outliers] temperature_2m_min: 0 outliers flagged (0.00%)
[flag_outliers] temperature_2m_mean: 0 outliers flagged (0.00%)
[flag_outliers] precipitation_sum: 1708 outliers flagged (18.47%)
[flag_outliers] wind_speed_10m_max: 166 outliers flagged (1.79%)
[flag_outliers] wind_gusts_10m_max: 172 outliers flagged (1.86%)
[flag_outliers] pressure_msl_mean: 30 outliers flagged (0.32%)
[flag_outli

c:\Users\Vito\Desktop\Heat-Pulse  clone\src\cleaning.py:119: RuntimeWarning: invalid value encountered in scalar divide
  print(f"[flag_outliers] {col}: {n_flagged} outliers flagged ({n_flagged/len(df)*100:.2f}%)")
c:\Users\Vito\Desktop\Heat-Pulse  clone\src\cleaning.py:119: RuntimeWarning: invalid value encountered in scalar divide
  print(f"[flag_outliers] {col}: {n_flagged} outliers flagged ({n_flagged/len(df)*100:.2f}%)")
c:\Users\Vito\Desktop\Heat-Pulse  clone\src\cleaning.py:119: RuntimeWarning: invalid value encountered in scalar divide
  print(f"[flag_outliers] {col}: {n_flagged} outliers flagged ({n_flagged/len(df)*100:.2f}%)")
c:\Users\Vito\Desktop\Heat-Pulse  clone\src\cleaning.py:119: RuntimeWarning: invalid value encountered in scalar divide
  print(f"[flag_outliers] {col}: {n_flagged} outliers flagged ({n_flagged/len(df)*100:.2f}%)")
c:\Users\Vito\Desktop\Heat-Pulse  clone\src\cleaning.py:119: RuntimeWarning: invalid value encountered in scalar divide
  print(f"[flag_outl

[validate_date_continuity] Ganja: Timeline is complete ✓
[validate_date_continuity] Mingachevir: Timeline is complete ✓
[validate_date_continuity] Sumqayit: Timeline is complete ✓
  Date-gap summary → DuckDB table: staging_historical_date_gaps
  DuckDB table 'staging_historical' written (9,248 rows)
  Parquet saved → c:\Users\Vito\Desktop\Heat-Pulse  clone\data\staging_historical.parquet

  FORECAST  →  staging_forecast
  Found parquet: c:\Users\Vito\Desktop\Heat-Pulse  clone\data\raw_forecast.parquet
  Rows: 0  |  Columns: ['time', 'city', 'latitude', 'longitude', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'pressure_msl_mean', 'shortwave_radiation_sum', 'apparent_temperature_max', 'weather_code']
  After cleaning: 0 rows
[flag_outliers] temperature_2m_max: 0.0 outliers flagged (nan%)
[flag_outliers] temperature_2m_min: 0.0 outliers flagged (nan%)
[flag_outliers] tempera

CSV not found — using fallback city list.


  Analytics table 'analytics_historical' written ✓

Building features: staging_forecast  →  analytics_forecast
  [SKIP] Could not load staging_forecast: Catalog Error: Table with name staging_forecast does not exist!
Did you mean "staging_historical"?

LINE 1: SELECT * FROM staging_forecast
                      ^

[populate_analytics_tables] Done ✓

───────────────────────────────────────────────────────────────────────────────────────────────
  CHECK                         STAGE                      STATUS  DETAILS
───────────────────────────────────────────────────────────────────────────────────────────────
  row_count                     raw_historical             ✅ PASS   9,248 rows present.
  freshness                     raw_historical             ✅ PASS   Data is fresh. Latest date: 2026-04-30 (lag 0 days).
  row_count                     staging_historical         ✅ PASS   9,248 rows present.
  null_ratio                    staging_historical         ✅ PASS   All numeric col

CSV not found — using fallback city list.



── Full run summary ──────────────────────
  status              : SUCCESS
  mode                : full
  rows_raw            : 9248
  rows_staging        : 9248
  rows_analytics      : 9248
  duration_sec        : 21.91
  quality_checks      :              check_name                 stage status  \
0             row_count        raw_historical   PASS   
1             freshness        raw_historical   PASS   
2             row_count    staging_historical   PASS   
3            null_ratio    staging_historical   PASS   
4       date_continuity    staging_historical   PASS   
5          value_ranges    staging_historical   PASS   
6             row_count  analytics_historical   PASS   
7  feature_completeness  analytics_historical   PASS   

                                             details  
0                                9,248 rows present.  
1  Data is fresh. Latest date: 2026-04-30 (lag 0 ...  
2                                9,248 rows present.  
3               All numeric c

## 2  Inspect database after full run

In [4]:
import duckdb
import pandas as pd

conn = duckdb.connect(str(DB_PATH))

from database import print_row_counts
print_row_counts(conn)

# Quick peek at analytics layer
try:
    df_analytics = conn.execute('SELECT * FROM analytics_historical LIMIT 5').df()
    print('analytics_historical columns:', list(df_analytics.columns))
    display(df_analytics)
except Exception as e:
    print(f'analytics_historical not available: {e}')

conn.close()



  VERİLƏNLƏR BAZASI — CƏDVƏLLƏRİN SƏTIR SAYI
  raw_historical                 9,248  █
  raw_forecast                       0  
  staging_historical             9,248  █
  staging_forecast                   0  
  analytics_historical           9,248  █
  analytics_forecast                 0  
  pipeline_runs                     18  

analytics_historical columns: ['time', 'city', 'latitude', 'longitude', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'pressure_msl_mean', 'shortwave_radiation_sum', 'apparent_temperature_max', 'weather_code', 'temperature_2m_mean_7d', 'temperature_2m_mean_30d', 'precipitation_sum_7d', 'precipitation_sum_30d', 'month', 'quarter', 'day_of_year', 'season', 'temperature_range', 'HDD', 'CDD', 'anomaly_score', 'temperature_2m_mean_lag1', 'temperature_2m_mean_lag2', 'precipitation_sum_lag1', 'precipitation_sum_lag2']


,time,city,latitude,longitude,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,...,day_of_year,season,temperature_range,HDD,CDD,anomaly_score,temperature_2m_mean_lag1,temperature_2m_mean_lag2,precipitation_sum_lag1,precipitation_sum_lag2
0,2020-01-01,Baku,40.4093,49.8671,11.4,4.4,6.6,0.0,0.0,0.0,...,1,winter,7.0,11.4,0.0,0.857143,NaN,NaN,NaN,NaN
1,2020-01-02,Baku,40.4093,49.8671,8.6,2.7,6.5,0.3,0.3,0.0,...,2,winter,5.9,11.5,0.0,0.028571,6.6,NaN,0.0,NaN
2,2020-01-03,Baku,40.4093,49.8671,7.6,3.4,5.6,1.4,1.4,0.0,...,3,winter,4.2,12.4,0.0,-0.671429,6.5,6.6,0.3,0.0
3,2020-01-04,Baku,40.4093,49.8671,8.4,5.7,6.8,0.2,0.2,0.0,...,4,winter,2.7,11.2,0.0,0.514286,5.6,6.5,1.4,0.3
4,2020-01-05,Baku,40.4093,49.8671,7.7,5.8,6.8,2.6,2.6,0.0,...,5,winter,1.9,11.2,0.0,0.171429,6.8,5.6,0.2,1.4


## 3  Incremental pipeline run

Equivalent to:
```bash
python src/pipeline.py --mode incremental
```

Because we just ran a full load, the pipeline should detect that all cities are
already up-to-date and skip the API fetch gracefully.


In [5]:
summary_inc = run_pipeline(
    mode     = 'incremental',
    db_path  = DB_PATH,
    data_dir = DATA_DIR,
    log_dir  = LOG_DIR,
)

print('\n── Incremental run summary ───────────────')
for k, v in summary_inc.items():
    print(f'  {k:<20}: {v}')

print(f'\nCities skipped (already up-to-date): {summary_inc.get("cities_skipped", 0)}')
print(f'New rows ingested: {summary_inc.get("rows_ingested", 0)}')


CSV not found — using fallback city list.
[Nakhchivan] Attempt 1 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=39.209&longitude=45.4122&daily=temperature_2m_max%2Ctemperature_2m_min%2Ctemperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Cwind_speed_10m_max%2Cwind_gusts_10m_max%2Cpressure_msl_mean%2Cshortwave_radiation_sum%2Capparent_temperature_max%2Cweather_code&timezone=UTC&start_date=2020-01-01&end_date=2026-04-30
[Nakhchivan] Attempt 2 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=39.209&longitude=45.4122&daily=temperature_2m_max%2Ctemperature_2m_min%2Ctemperature_2m_mean%2Cprecipitation_sum%2Crain_sum%2Csnowfall_sum%2Cwind_speed_10m_max%2Cwind_gusts_10m_max%2Cpressure_msl_mean%2Cshortwave_radiation_sum%2Capparent_temperature_max%2Cweather_code&timezone=UTC&start_date=2020-01-01&end_date=2026-04-30
[Nakhchivan] Attempt 3 failed: 429 Client Error:


  Yeni məlumat yoxdur — baza artıq aktual idi.

───────────────────────────────────────────────────────────────────────────────────────────────
  CHECK                         STAGE                      STATUS  DETAILS
───────────────────────────────────────────────────────────────────────────────────────────────
  row_count                     raw_historical             ✅ PASS   9,248 rows present.
  freshness                     raw_historical             ✅ PASS   Data is fresh. Latest date: 2026-04-30 (lag 0 days).
  row_count                     staging_historical         ✅ PASS   9,248 rows present.
  null_ratio                    staging_historical         ✅ PASS   All numeric columns have ≤ 5% nulls.
  date_continuity               staging_historical         ✅ PASS   No gaps > 3 days found in any city.
  value_ranges                  staging_historical         ✅ PASS   All temperature values within [-50.0°C, 60.0°C].
  row_count                     analytics_historical       ✅ 

## 4  Quality gate results (standalone demo)

In [6]:
import duckdb
from quality_checks import run_all_checks, print_quality_report

conn = duckdb.connect(str(DB_PATH))

try:
    raw_df      = conn.execute('SELECT * FROM raw_historical').df()
except Exception:
    raw_df = None

try:
    staging_df  = conn.execute('SELECT * FROM staging_historical').df()
except Exception:
    staging_df = None

try:
    analytics_df = conn.execute('SELECT * FROM analytics_historical').df()
except Exception:
    analytics_df = None

conn.close()

results = run_all_checks(
    raw_df       = raw_df,
    staging_df   = staging_df,
    analytics_df = analytics_df,
)
print_quality_report(results)

# Display as a DataFrame for notebook clarity
pd.DataFrame(results)



───────────────────────────────────────────────────────────────────────────────────────────────
  CHECK                         STAGE                      STATUS  DETAILS
───────────────────────────────────────────────────────────────────────────────────────────────
  row_count                     raw_historical             ✅ PASS   9,248 rows present.
  freshness                     raw_historical             ✅ PASS   Data is fresh. Latest date: 2026-04-30 (lag 0 days).
  row_count                     staging_historical         ✅ PASS   9,248 rows present.
  null_ratio                    staging_historical         ✅ PASS   All numeric columns have ≤ 5% nulls.
  date_continuity               staging_historical         ✅ PASS   No gaps > 3 days found in any city.
  value_ranges                  staging_historical         ✅ PASS   All temperature values within [-50.0°C, 60.0°C].
  row_count                     analytics_historical       ✅ PASS   9,248 rows present.
  feature_completenes

,check_name,stage,status,details
0,row_count,raw_historical,PASS,"9,248 rows present."
1,freshness,raw_historical,PASS,Data is fresh. Latest date: 2026-04-30 (lag 0 ...
2,row_count,staging_historical,PASS,"9,248 rows present."
3,null_ratio,staging_historical,PASS,All numeric columns have ≤ 5% nulls.
4,date_continuity,staging_historical,PASS,No gaps > 3 days found in any city.
5,value_ranges,staging_historical,PASS,"All temperature values within [-50.0°C, 60.0°C]."
6,row_count,analytics_historical,PASS,"9,248 rows present."
7,feature_completeness,analytics_historical,PASS,All 16 required feature columns present and no...


## 5  View pipeline log

In [7]:
log_file = LOG_DIR / 'pipeline.log'

if log_file.exists():
    lines = log_file.read_text(encoding='utf-8').splitlines()
    # Show last 40 lines
    print(f'... (showing last 40 of {len(lines)} lines) ...\n')
    print('\n'.join(lines[-40:]))
else:
    print(f'Log file not found at {log_file}')


... (showing last 40 of 2708 lines) ...

2026-04-30 16:17:16,881 | INFO     | ingestion | [zardab] Fetching 2026-04-01 → 2026-04-30 (attempt 1)
2026-04-30 16:17:17,951 | INFO     | ingestion | [zardab] Fetched 30 rows.
2026-04-30 16:17:17,952 | INFO     | ingestion | [zardab] 30 new rows (2026-04-01 → 2026-04-30).
2026-04-30 16:17:17,963 | INFO     | ingestion | Incremental ingest: 2,820 new rows total.
2026-04-30 16:17:17,964 | INFO     | pipeline | Məlumat alındı: 2,820 sətir
2026-04-30 16:17:17,965 | INFO     | pipeline | === MƏRHƏLƏ 2: XAM MƏLUMAT YÜKLƏNIR ===
2026-04-30 16:17:18,007 | INFO     | database | raw_historical-a 2,820 sətir yükləndi.
2026-04-30 16:17:18,389 | INFO     | database | 216,962 sətir saxlandı → data\raw.parquet
2026-04-30 16:17:18,774 | INFO     | database | 216,962 sətir saxlandı → data\raw_historical.parquet
2026-04-30 16:17:18,804 | INFO     | database | 0 sətir saxlandı → data\raw_forecast.parquet
2026-04-30 16:17:18,927 | INFO     | quality_checks | [row

## 6  Pipeline Architecture Diagram

```
╔══════════════════════════════════════════════════════════════════════════╗
║             HEAT-PULSE  —  End-to-End Pipeline Architecture              ║
╚══════════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────────┐
  │                        TRIGGER (CLI)                                │
  │   python src/pipeline.py --mode [full | incremental]                │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 1 — INGEST            (src/ingestion.py + pipeline.py)       │
  │                                                                     │
  │  full mode       → fetch ALL dates from Open-Meteo API              │
  │  incremental     → check max(time) per city in raw_historical        │
  │                    fetch only dates AFTER that                      │
  │                    APPEND (not replace) new rows                    │
  │                                                                     │
  │  Output → DuckDB: raw_historical table                              │
  │         → data/raw.parquet  (file-system backup)                    │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #1           │
                    │  ✓ row_count  (FAIL→abort) │
                    │  ✓ freshness  (WARN)       │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 2 — CLEAN             (src/cleaning.py)                      │
  │                                                                     │
  │  handle_missing_values()  → ffill temp, zero precip, interpolate    │
  │  flag_outliers()          → IQR flags, rows kept                    │
  │  validate_date_continuity()→ gap audit per city                     │
  │                                                                     │
  │  Output → DuckDB: staging_historical table                          │
  │         → data/staging_historical.parquet                           │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #2           │
                    │  ✓ null_ratio    (WARN)    │
                    │  ✓ date_contin.  (WARN)    │
                    │  ✓ value_ranges  (WARN)    │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  STAGE 3 — FEATURES          (src/features.py)                      │
  │                                                                     │
  │  add_rolling_averages()   → 7d / 30d means                          │
  │  add_seasonal_indicators()→ month, quarter, season                  │
  │  add_temperature_range()  → max − min                               │
  │  add_degree_days()        → HDD / CDD @ 18°C base                  │
  │  add_anomaly_score()      → deviation from DOY mean                 │
  │  add_lag_features()       → lag-1 / lag-2 temp & precip             │
  │                                                                     │
  │  Output → DuckDB: analytics_historical table                        │
  └───────────────────────────────┬─────────────────────────────────────┘
                                  │
                    ┌─────────────▼─────────────┐
                    │  QUALITY GATE #3           │
                    │  ✓ feature_completeness    │
                    │    (WARN if cols missing)  │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
  ┌─────────────────────────────────────────────────────────────────────┐
  │  LOGGING  (logs/pipeline.log)                                       │
  │  • Start / end time & total duration                                │
  │  • Rows per stage per city                                          │
  │  • Errors & warnings                                                │
  │  • All quality gate results                                         │
  └─────────────────────────────────────────────────────────────────────┘

  Data layers in DuckDB
  ─────────────────────
  raw_historical       ← untouched API responses
  staging_historical   ← cleaned, nulls handled, outliers flagged
  analytics_historical ← ML-ready features added
```


## 7  CLI equivalents (for reference)

You can reproduce everything in this notebook from your terminal:

```bash
# Full historical re-ingest
python src/pipeline.py --mode full

# Incremental (only new days)
python src/pipeline.py --mode incremental

# Incremental for specific cities only
python src/pipeline.py --mode incremental --cities Baku Ganja

# Custom paths
python src/pipeline.py --mode full --db-path data/weather.duckdb --data-dir data --log-dir logs
```
